# vowl · Results Tour

A finished validation is only useful once you can *read* it. This notebook is a
tour of every shape a `ValidationResult` can take once the checks have run,
using the HDB Resale dataset that ships with the repo (a copy seeded with
deliberate errors so some checks fail and there is something to look at).

Everything here reads an already-finished result. Nothing is re-run against your
data between sections, and each output is just a different view of the same run.

What this notebook covers:

1. Setup and a validation run to work with
2. **Annotated output**: your full table with a `check_info` column, flagged rows first
3. **Residual rows**: failed rows for checks that cannot attach to a single table
4. **Saving to disk**: the annotated tables and residues as CSV plus a JSON summary

Everything here needs only a base `vowl` install. The run's DQ metrics, as
`dq_metrics.json` and as OpenTelemetry, have their own notebook in
[`6_dq_metrics/`](../6_dq_metrics/dq_metrics.ipynb). The files this notebook
writes land in the `outputs/` folder beside it, kept in the repo as reference so
you can see the shapes without running anything.

<a id="1-setup"></a>
## 1. Setup and a validation run

Resolve the dataset paths, load the HDB CSV, and run `validate_data`. Some checks
fail, which is exactly what we want: the outputs below are only interesting when
there is something flagged.

In [1]:
# Install vowl (skipped during execution)
# %pip install vowl

In [2]:
from pathlib import Path

# Walk up to the repo root (works no matter how deep this notebook sits)
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "tests" / "hdb_resale").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# Single-source: HDB Resale dataset (used for sections 2 and 4)
HDB_DIR = REPO_ROOT / "tests" / "hdb_resale"
HDB_CSV = HDB_DIR / "HDBResaleWithErrors.csv"
HDB_CONTRACT = HDB_DIR / "hdb_resale_simple.yaml"  # Switch to "hdb_resale.yaml" for a more complex contract

# Multi-source: Employee dataset (payroll + employee list), used for residues in section 3
EMPLOYEE_DIR = REPO_ROOT / "tests" / "employee"
EMPLOYEE_PAYROLL_CSV = EMPLOYEE_DIR / "demo_employee_payroll.csv"
EMPLOYEE_LIST_CSV = EMPLOYEE_DIR / "demo_employee_list.csv"
EMPLOYEE_CONTRACT = EMPLOYEE_DIR / "employee_payroll_datacontract.yaml"

print(f"HDB CSV exists:      {HDB_CSV.exists()}")
print(f"HDB contract exists: {HDB_CONTRACT.exists()}")

HDB CSV exists:      True
HDB contract exists: True


In [3]:
import warnings

import pandas as pd

from vowl import validate_data

# vowl surfaces informational UserWarnings on this demo dataset (Arrow coercion,
# adapter reuse). Quieten them so the output below stays readable.
warnings.filterwarnings("ignore", category=UserWarning, module=r"vowl\.validation\.runner")

df = pd.read_csv(HDB_CSV, low_memory=False).fillna("")
result = validate_data(contract=str(HDB_CONTRACT), df=df)

print(f"passed:        {result.passed}")
print(f"total checks:  {result._vs.get('total_checks')}")
print(f"failed:        {result._vs.get('failed')}")
print(f"errors:        {result._vs.get('errors')}")

passed:        False
total checks:  9
failed:        2
errors:        0


<a id="2-annotated"></a>
## 2. Annotated Output: Your Full Table, Flagged

The main accessor here is **`get_annotated_output()`**. It returns your full table with one extra column, `check_info`, so you can see failures in the context of every row instead of on their own.

It returns a dict with two keys:

- **`"annotated"`**: `{schema: table}`. Each table is your full data plus a `check_info` column. Every original row is kept. `check_info` is `null` for rows that passed every check, and a list of the failing check(s) for rows that didn't.
- **`"residues"`**: failed rows for checks that can't be attached to a single table (cross-table, aggregation, and column-subset checks). The single-table HDB contract here has none; [section 3](#3-residual-non-mergeable-rows) shows them using a multi-source contract.

> Not every check can be merged into the annotated table. For the full rules and worked examples, see [Known Issues: Annotated Output](../../docs/known-issues.md#annotated-output-not-all-checks-can-be-merged).

### Choosing How Much Detail: the `check_info` Options

`check_info` is a **list of objects, one per failing check** (stored as a JSON string in the column). The `check_info=` argument sets how many fields each object carries:

| `check_info` | Each object contains | Use it when |
|---|---|---|
| `"names"` *(default)* | `check_name` | You only need to know *which* checks a row failed |
| `"summary"` | `check_name`, `dimension`, `tags`, `target` | You want each failure's quality dimension, tags, and column (e.g. to split completeness vs. accuracy) |
| `"full"` | the entire check definition (`type`, `description`, `query`, `mustBe`, `dimension`, `tags`) plus `check_name` and `target` | You want everything, including the underlying rule/SQL, on every row |

All three return the **same shape** (a list of objects) so you always read a value the same way (`json.loads(cell)[0]["check_name"]`); they differ only in how many fields each object has. A row failing two checks gets a two-object list.

```text
"names"    [{"check_name": "Month"}]
"summary"  [{"check_name": "Month", "dimension": "conformity",
             "tags": null, "target": "hdb_resale_prices.month"}]
"full"     [{"type": "sql", "name": "Month", "description": "...", "mustBe": 0,
             "query": "SELECT COUNT(*) ...",
             "dimension": "conformity", "check_name": "Month",
             "target": "hdb_resale_prices.month"}]
```

The examples below use `"summary"`.

In [4]:
# "annotated" -> {schema: your full table + a check_info column}
# "residues"  -> failed rows for checks that can't attach to one table (shown later)
# check_info="summary" -> each object also carries dimension, tags, and target.
output = result.get_annotated_output(check_info="summary")
annotated = output["annotated"]["hdb_resale_prices"].to_pandas()

In [5]:
# Inspect the structure of the annotated output
print("Top-level keys:    ", list(output.keys()))
print("Annotated schemas: ", list(output["annotated"].keys()))
print("Residue keys:      ", list(output["residues"].keys()))
print(f"\nAnnotated table: {annotated.shape[0]:,} rows x {annotated.shape[1]} columns")
print("Columns:", list(annotated.columns))

Top-level keys:     ['annotated', 'residues']
Annotated schemas:  ['hdb_resale_prices']
Residue keys:       []

Annotated table: 201,879 rows x 12 columns
Columns: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price', 'check_info']


### Inspecting the Full Table, Flagged Rows First

Every original row is present. `check_info` is `null` for rows that passed and a JSON list for rows that failed, one object per failing check (here carrying `dimension`, `tags`, and `target` from the `"summary"` preset). Sorting by `check_info` (nulls last) floats the flagged rows to the top while still showing the whole table.

In [6]:
# Sort the full table so flagged rows surface first; passing rows (check_info = null) sink.
flagged_first = annotated.sort_values("check_info", na_position="last").reset_index(drop=True)
n_flagged = annotated["check_info"].notna().sum()

In [7]:
print(f"{n_flagged} flagged rows at the top, {len(annotated) - n_flagged:,} passing rows below\n")
flagged_first[["month", "town", "block", "floor_area_sqm",
               "lease_commence_date", "check_info"]].head(20)

14 flagged rows at the top, 201,865 passing rows below



,month,town,block,floor_area_sqm,lease_commence_date,check_info
0,2017-jan,BEDOK,21,130.0,1972,"[{""check_name"": ""Month"", ""dimension"": ""conform..."
1,2017-jan,BISHAN,105,4.0,1985,"[{""check_name"": ""Month"", ""dimension"": ""conform..."
2,2017-06,KALLANG/WHAMPOA,38,215.0,1972,"[{""check_name"": ""floor_area_must_be_less_than_..."
3,2017-09,CHOA CHU KANG,641,215.0,1998,"[{""check_name"": ""floor_area_must_be_less_than_..."
4,2017-12,KALLANG/WHAMPOA,65,249.0,1972,"[{""check_name"": ""floor_area_must_be_less_than_..."
5,2018-01,CHOA CHU KANG,639,215.0,1998,"[{""check_name"": ""floor_area_must_be_less_than_..."
6,2018-09,KALLANG/WHAMPOA,41,237.0,1972,"[{""check_name"": ""floor_area_must_be_less_than_..."
7,2021-03,BISHAN,446,243.0,1990,"[{""check_name"": ""floor_area_must_be_less_than_..."
8,2021-04,BISHAN,454,243.0,1989,"[{""check_name"": ""floor_area_must_be_less_than_..."
9,2021-04,KALLANG/WHAMPOA,61,222.0,1972,"[{""check_name"": ""floor_area_must_be_less_than_..."


### Keeping the Clean Rows for Downstream Use

Because the annotation lives on the full table, separating the good rows from the bad is a one-liner. Filter to where `check_info` is null, drop the annotation column, and you have a clean dataset ready to feed downstream -- no separate join back to the source needed.

In [8]:
# Keep only the rows that passed every check, then drop the annotation column
clean = annotated[annotated["check_info"].isna()].drop(columns=["check_info"])

In [9]:
print(f"Clean rows ready for downstream use: {len(clean):,} of {len(annotated):,}")
clean.head()

Clean rows ready for downstream use: 201,865 of 201,879


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0


<a id="3-residues"></a>
<a id="residual-rows"></a>
## 3. Residual (Non-Mergeable) Rows

The HDB examples above are single-table, so `residues` is empty. Residues appear when a check can't be attached to one table: **aggregation** checks, **column-subset** checks, and **cross-table** checks whose failed rows carry columns from more than the anchor table.

Residues are **per-check**: `get_annotated_output()` returns one entry for each non-mergeable check, keyed `"<schema>::<check_name>"`. They are never grouped together, so a row that fails two such checks appears once under each check's entry. Each entry carries the failed rows plus the same `check_info` column the annotated tables use (a single-element JSON array, shaped by the `check_info` preset) and `tables_in_query`, so everything `get_annotated_output()` returns is read the same way.

> **A cross-table check can *merge* instead of becoming a residue.** If you shape its row query to project only the anchor table's columns (e.g. `SELECT payroll.*` inside a subquery), the orphan rows match that schema and land directly in its `check_info` column, with no residue. The Employee contract below carries both shapes: `orphan_payroll_rows_merge_onto_payroll` (subquery-projected, merges onto `demo_employee_payroll`) and `employee_id_exists_in_master_list` / `phone_number_exists_in_master_list` (bare JOINs, stay residues). See [Known Issues: Annotated Output](../../docs/known-issues.md#annotated-output-not-all-checks-can-be-merged) for the rules.

Below is a quick multi-source run on the Employee dataset, whose contract has cross-table checks. (Multi-source validation is covered in the [Multiple Sources notebook](../2_multiple_sources/multiple_sources.ipynb); we borrow it here just to produce residues.)

In [10]:
import ibis
from vowl.adapters import IbisAdapter

# Load the two Employee tables onto one DuckDB connection (cross-table contract)
con = ibis.duckdb.connect()
con.create_table("demo_employee_payroll", pd.read_csv(EMPLOYEE_PAYROLL_CSV))
con.create_table("demo_employee_list", pd.read_csv(EMPLOYEE_LIST_CSV))

mt_result = validate_data(contract=str(EMPLOYEE_CONTRACT), adapter=IbisAdapter(con))
mt_output = mt_result.get_annotated_output()

In [11]:
# Each residue is the failed rows for ONE non-mergeable check, keyed "<schema>::<check>"
print("Annotated schemas:", list(mt_output["annotated"].keys()))
print("Residue keys:     ", list(mt_output["residues"].keys()))

for key, residue in mt_output["residues"].items():
    residue_df = residue.to_pandas()
    print(f"\nResidue '{key}': {len(residue_df)} failed row(s)")
    display(residue_df[["employee_id", "payroll_id", "month",
                        "check_info", "tables_in_query"]])

Annotated schemas: ['demo_employee_payroll', 'demo_employee_list']
Residue keys:      []


### The Other Side: a Cross-Table Check That *Merges*

Whether a cross-table check merges is decided entirely by **the columns its failed rows come back with**: they have to match the anchor table's columns exactly. The two checks above join `payroll` against a reference table with a bare `SELECT *`, so their failed rows carry columns from *both* tables and can't land on any single table, so they become a residue.

The *same* referential question **merges** when you wrap it so the inner query projects only the payroll columns (`SELECT payroll.* ...`). The contract's `orphan_payroll_rows_merge_onto_payroll` check does exactly that, so its failed rows come back with exactly the payroll columns and are annotated **directly onto the payroll table's `check_info` column** (notice it's absent from the residue keys above).

> For the full mechanics (how vowl derives the scalar and row queries, and why the `COUNT(*)` -> `SELECT *` rewrite only touches the outer projection), see [Known Issues: Cross-table checks, mergeable when the failed rows match the home schema](../../docs/known-issues.md#1-cross-table-checks-mergeable-when-the-failed-rows-match-the-home-schema).

In [12]:
# The subquery-projected cross-table check lands on the payroll annotated table,
# not in residues. Find the payroll rows it flagged.
payroll_annotated = mt_output["annotated"]["demo_employee_payroll"].to_pandas()

MERGED_CHECK = "orphan_payroll_rows_merge_onto_payroll"
flagged = payroll_annotated[
    payroll_annotated["check_info"].fillna("").str.contains(MERGED_CHECK)
]

print(f"'{MERGED_CHECK}' in residues? "
      f"{any(MERGED_CHECK in k for k in mt_output['residues'])}")
print(f"Payroll rows it annotated: {len(flagged)}\n")
display(flagged[["employee_id", "month", "check_info"]])

'orphan_payroll_rows_merge_onto_payroll' in residues? False
Payroll rows it annotated: 1



,employee_id,month,check_info
2,e939123,2025-12,"[{""check_name"": ""phone_number_logical_type_opt..."


<a id="4-saving"></a>
## 4. Saving Outputs to Disk

`result.save(...)` writes the annotated tables by default. The `output_mode` argument controls the layout. Every mode also writes `<prefix>_check_results.csv` and `<prefix>_summary.json`, and the annotated modes also write the run's [DQ metrics](../6_dq_metrics/dq_metrics.ipynb) as `<prefix>_dq_metrics.json`:

| Mode | Files written |
|------|----------------|
| `"failed_rows"` *(cheap)* | One grouped failed-rows CSV per table key, e.g. `<prefix>_<table>.csv`. No `dq_metrics.json`. |
| `"annotated"` *(default)* | One `<prefix>_<schema>_annotated.csv` per schema (full table + `check_info`), **plus** one `<prefix>_<schema>_<check>_residue.csv` per non-mergeable check, **plus** `<prefix>_dq_metrics.json` |
| `"both"` | Everything `"annotated"` writes, residues included, **plus** the failed-rows CSVs |

The `check_info` argument (`"names"` / `"summary"` / `"full"`, described above) sets how much detail the saved `check_info` column carries. To set the `check_info` detail for every `save()`, pass `ValidationConfig(annotated_check_info="summary")` to `validate_data(config=...)`.

The annotated modes attribute failed rows to each table and may download it, which can be slow on a large table. The annotated tables and `dq_metrics.json` share that work, so writing both costs no more than writing one. `"failed_rows"` runs no extra queries and never downloads a table, so pick it when you only need the failed rows.

Single-table contracts (like the HDB example) have no non-mergeable checks, so they never produce `_residue` files. The multi-source run below does, so let's save both to see the difference. The files land in the `outputs/` folder beside this notebook.

In [13]:
# Write the annotated table(s) to disk
result.save(output_dir="outputs", prefix="vowl_demo_annotated", output_mode="annotated")


Results saved:
   - outputs/vowl_demo_annotated_check_results.csv
   - outputs/vowl_demo_annotated_hdb_resale_prices_annotated.csv
   - outputs/vowl_demo_annotated_summary.json
   - outputs/vowl_demo_annotated_dq_metrics.json


ValidationResult(passed=False, checks=9, passed_checks=7, failed_checks=2)

### Saving a Run That Has Residues

`result` above is single-table, so its `save(output_mode="annotated")` wrote only `*_annotated.csv` files. Saving the multi-source `mt_result` from the residue example instead also writes one `*_residue.csv` per non-mergeable check, keyed `<schema>_<check>` (the `::` in the residue key becomes `_`). Each residue CSV carries that one check's failed rows plus the same `check_info` column as the annotated tables and `tables_in_query`.

In [14]:
import os

# Save the multi-source run; annotated mode emits residue CSVs for the
# cross-table checks that can't be merged onto a single table.
mt_result.save(output_dir="outputs", prefix="vowl_demo_residues", output_mode="annotated")

print("\nFiles written:")
for fname in sorted(f for f in os.listdir("outputs") if f.startswith("vowl_demo_residues")):
    tag = "  <- residue" if fname.endswith("_residue.csv") else ""
    print(f"   {fname}{tag}")


Results saved:
   - outputs/vowl_demo_residues_check_results.csv
   - outputs/vowl_demo_residues_demo_employee_payroll_annotated.csv
   - outputs/vowl_demo_residues_demo_employee_list_annotated.csv
   - outputs/vowl_demo_residues_summary.json
   - outputs/vowl_demo_residues_dq_metrics.json

Files written:
   vowl_demo_residues_check_results.csv
   vowl_demo_residues_demo_employee_list_annotated.csv
   vowl_demo_residues_demo_employee_payroll_annotated.csv
   vowl_demo_residues_demo_employee_payroll_employee_id_exists_in_master_list_residue.csv  <- residue
   vowl_demo_residues_demo_employee_payroll_phone_number_exists_in_master_list_residue.csv  <- residue
   vowl_demo_residues_dq_metrics.json
   vowl_demo_residues_summary.json


### Where the Full Check Definitions Live

To keep the annotated table readable, you'll usually pick `"names"` or `"summary"` rather than `"full"`. You don't lose anything: the complete definition of **every** check (its `dimension`, `tags`, `description`, `query`, and more) is always written to the `*_summary.json` file that `save()` produces, keyed by check name. Keep the table light, and look up the rest here when you need it.

In [15]:
import json

# save(prefix="vowl_demo_annotated", ...) wrote this alongside the CSVs.
with open("outputs/vowl_demo_annotated_summary.json") as f:
    summary = json.load(f)

# Every check's resolved definition lives under "check_results", keyed by name.
month_check = next(c for c in summary["check_results"] if c["name"] == "Month")

print("Target:    ", month_check["target"])
print("Dimension: ", month_check["check_definition"]["dimension"])
print("\nFull check_definition:")
print(json.dumps(month_check["check_definition"], indent=2))

Target:     hdb_resale_prices.month
Dimension:  conformity

Full check_definition:
{
  "type": "sql",
  "name": "Month",
  "description": "Based on ISO 8601 | YYYY-MM",
  "mustBe": 0,
  "query": "SELECT COUNT(*)\nFROM \"hdb_resale_prices\"\nWHERE CAST(month AS TEXT) !~ '^[0-9]{4}-(0[1-9]|1[0-2])$';",
  "dimension": "conformity"
}


---

### Recap

- `get_annotated_output()` returns your full table with a `check_info` column, so
  flagged rows read in context and the clean rows fall out with a single filter.
- Checks that can't attach to one table come back as **residues**, one entry per
  non-mergeable check.
- `result.save(output_mode="annotated")` writes those same shapes to disk as CSV,
  plus a `*_summary.json` that always carries every full check definition and a
  `*_dq_metrics.json` with the run's DQ metrics.
- The `outputs/` folder holds a saved example of each output for reference.

Next, [`6_dq_metrics/dq_metrics.ipynb`](../6_dq_metrics/dq_metrics.ipynb) reads the
same run as DQ metrics: the `*_dq_metrics.json` file written above, and the
OpenTelemetry export for your monitoring tools.